# Research Log: Framing the Game

Running notebook for brainstorms, hypotheses, experimental results, and reviewer-driven insights.

**Paper**: *Framing the Game: How Context Shapes LLM Decision-Making*  
**Authors**: Isaac Robinson, John Burden  
**Target**: ICLR 2026 resubmission  

---

## Key Reviewer Feedback (ICLR 2026 Round 1)

**Result**: Reject (scores: 4, 2, 6, 4)

### Must-address concerns
1. **Generalizability beyond one-shot PD** — all four reviewers flagged this. Need to either extend to other games or better justify the PD-only scope.
2. **LLM-as-judge QC without human validation** — meta-reviewer listed this as a primary rejection reason. Need human eval or at minimum judge-swap analysis.
3. **Descriptive, not mechanistic** — reviewers wanted to know *why* certain contexts drive cooperation, not just *that* they do (mVh3, aAtL).
4. **Limited reasoning model coverage** — mQG5 specifically flagged this; only 2 small R1 distillations tested.

### Addressable in resubmission
- Better topic selection to tell a clearer story
- 108-model registry now covers full reasoning model spectrum (o1, o3, R1, QwQ)
- Need behavioral/mechanistic analysis connecting topics to cooperation rates
- Need human validation study for vignette QC

---

## Brainstorms & Hypotheses

*(newest first)*

### 2026-02-20: Topic Redesign Brainstorm

**Problem**: Original 10 topics were redundant (5/10 politics, 2 generic) and produced results that were descriptive but not compelling. Reviewers asked "why" and we couldn't answer.

**Insight**: Topics should be chosen so the cooperation gradient is *predictable a priori* from domain norms, making the finding interpretable rather than just descriptive.

#### Promising axes of variation

1. **Moral valence of cooperation** — cooperation can be prosocial (sharing), neutral (trade), or antisocial (collusion, cover-up). If models cooperate less when cooperation is "wrong", that's evidence of learned moral reasoning overriding game theory.

2. **Cultural/geographic framing** — same scenario with different cultural context (Silicon Valley vs Tokyo vs Lagos). Would reveal training-data cultural stereotypes in strategic behavior. High bias/fairness relevance.

3. **Power asymmetry** — large corp vs startup, employer vs employee. Tests whose interests models default to.

4. **Observability** — private vs public negotiation. In one-shot PD, observability shouldn't matter. If it does, models are importing reputation-game logic.

5. **Stakes magnitude** — neighborhood dispute vs international crisis. Tests stake-sensitivity.

6. **Political dyads** — specific party matchups (R vs D, D vs Green, bipartisan committee). Tests whether models reflect real-world partisan polarization.

7. **Temporal/historical distance** — same dilemma in 1400 vs 1940 vs 2025. Models may grant "moral license" to defect in historical settings.

#### Open question
Which 2-3 axes to combine? The interaction effects (e.g., "cultural framing matters more in business than medicine") are where the paper gets strongest.

### 2026-02-20: Axes Selected [design]

**Selected 6 axes** (dropped stakes magnitude):
1. **Moral valence of cooperation** — prosocial vs neutral vs antisocial cooperation
2. **Cultural/geographic framing** — same scenario, different cultural context
3. **Power asymmetry** — symmetric vs asymmetric actor power
4. **Observability** — private vs public (shouldn't matter in one-shot PD but probably does)
5. **Political dyads** — specific party matchups with known real-world antagonism levels
6. **Temporal/historical distance** — same dilemma across time periods

**Next step**: Combine into a concrete topic set (10-13 topics) that covers multiple axes without being unwieldy. Key challenge is that 6 axes fully crossed would be enormous — need to pick topics that naturally embed 1-2 axes each.

### 2026-02-20: Power Analysis Results [result]

**Setup**: Two-proportion z-test, α=0.05, power=0.80, baseline coop ~55%

**Key takeaways**:
- To detect a **30pp difference** (e.g. moral valence extremes): only **~42 stories/group** needed — very cheap
- To detect a **15pp difference** (e.g. political dyads, temporal): **~170 stories/group** — moderate
- To detect a **10pp difference** (e.g. cultural framing): **~390 stories/group** — expensive
- To detect a **5pp difference**: ~1,550/group — probably not worth it

**Implications for axis selection**:
- **Moral valence** (expected 30pp effect): easiest to power, ~126 stories total. Do this.
- **Political dyads** (expected 15-25pp): ~250-700 total across 4 groups. Feasible.
- **Temporal distance** (expected 10-20pp): ~290-1,164 total. Feasible if effect is large enough.
- **Observability** (binary, expected 10-20pp): ~192-540 total. Cheap because only 2 groups.
- **Power asymmetry** (binary, expected 10-20pp): ~194-540 total. Same.
- **Cultural/geographic** (expected 5-15pp): **riskiest** — if effect is only 5-10pp, need 1,500+ stories/group. Could be underpowered unless effect is surprisingly large.

**Budget**: At $0.62/sweep for all 108 models, 200 stories/group × 10 groups = $1,240 total. Manageable.

### 2026-02-20: Budget approved [design]

**$1,200 budget approved** for full 108-model sweep at ~200 stories/group × ~10 groups. All 6 axes are a go including cultural/geographic (will accept risk of underpowering if effect is small).

**Next step**: Design the concrete topic set and config structure — which axes become topics vs config dimensions.

### 2026-02-20: Experimental Design Decisions [design]

**Structure**: topics × actor_types × observability × power_dynamic

**Decisions made**:
- **Drop `neutral` actor type** — keep only allies vs enemies for cleaner contrast
- **Drop `world_type`** (real vs imaginary) — wasn't a strong finding in v1, doubles cell count for little value
- **Add `observability`**: private vs public (binary)
- **Add `power_dynamic`**: symmetric vs asymmetric (binary)
- **Budget**: uncapped for now, prioritize at least 6 topics per axis category
- **Target**: ≥6 topics per axis (moral valence, political dyads, temporal, cultural/geographic)

**Cross product**: N topics × 2 actors × 2 observability × 2 power = N × 8 cells per topic

### 2026-02-20: External Brainstorm Synthesis [brainstorm]

**Key refinements from external model feedback:**

1. **Temporal axis**: Hold the *type* of interaction constant (e.g., "two powers negotiating a trade pact"), vary ONLY time period. Avoids confounding substance with era.
2. **Moral valence**: Ensure antisocial cooperation is explicitly illegal/exploitative, not just morally gray. Clean 3-bin structure (prosocial / neutral / antisocial).
3. **Political dyads**: Expand to include international adversaries (US vs China, US vs EU) for a fuller ideological-distance gradient. Monotonic prediction: cooperation ∝ perceived affinity.
4. **Cultural**: Structure by Hofstede-style dimensions (collectivist vs individualist, high-trust vs low-trust). Add Nordic (Stockholm) for high-trust baseline.
5. **Orthogonality**: Don't let moral valence confound actor_type axis. "Rival gangs" already implies enemies + antisocial — avoid this.
6. **Track refusals**: Model refusing to engage with antisocial cooperation is itself a finding.
7. **Explicit hypotheses**: H1 (moral valence), H2 (ideological distance), H3 (temporal modernity), H4 (cultural trust proxies).
8. **Optional 5th axis**: Anthropomorphism (individuals vs corporations vs AI systems vs governments) — interesting but parking for now.

### 2026-02-20: Temporal axis redesign — zoom into modernity [design]

**Insight**: The interesting question isn't "ancient vs modern" — it's whether models reflect shifting norms *within* recent history. Training data is 1000x denser for 2000s-2020s than for 1200 BCE. Cultural inflection points (social media, 2016 polarization, COVID) may produce measurable cooperation shifts even across single decades.

**New design**: 1 ancient anchor + dense modern granularity. Same underlying scenario: "two sovereign powers negotiating a trade agreement."

Proposed eras:
- ~1200 BCE (Bronze Age anchor — Hobbesian baseline)
- 1850s (Industrial, pre-world-wars, imperial competition)
- 1940s (WWII/postwar, institutional cooperation born out of catastrophe)
- 1970s (détente, Cold War thaw, pragmatic cooperation)
- early 2000s (post-9/11, "war on terror," multilateralism strained)
- 2010s (pre-Trump, peak globalization consensus)  
- 2020s (COVID, polarization peak, institutional distrust)
- near-future (speculative — tests whether models default to optimism or dystopia)

**Hypothesis refinement**: Cooperation may NOT be monotonically increasing with time. Possible non-linear pattern:
- 1940s bump (postwar institution-building)
- 2000s dip (post-9/11 unilateralism)
- 2020s dip (polarization, distrust)
- Near-future uncertain (utopian vs dystopian training data)

This non-linearity would be a much more interesting finding than a simple "modern = more cooperative" gradient.

### 2026-02-20: Temporal axis — tighter parallel structure [design]

**Problem**: Current temporal topics vary both the era AND the scenario type (grain trade, rail access, reconstruction aid, pandemic response). This confounds time with substance — if cooperation differs between "1940s reconstruction aid" and "2020s pandemic response," is that because of the era or because aid ≠ pandemic?

**Fix**: Use the exact same scenario template for all time periods. Only the date and period-appropriate nouns change. E.g., "two leaders negotiating a trade agreement in [era]" — same actors (political leaders), same action (trade negotiation), same stakes.

### 2026-02-21: Design rigor audit [design]

Reviewing the 30-topic design for confounds, missing controls, and methodological gaps before implementation.

#### Issues identified:

**1. Moral valence axis is confounded by domain**
"Hospitals sharing ventilators" vs "banks manipulating interest rates" differs in moral valence AND domain, actors, stakes, and familiarity. Any cooperation difference could be healthcare vs finance, not prosocial vs antisocial. 

**Fix**: Use matched pairs within the SAME domain where cooperation flips moral meaning:
- "Two pharma companies sharing drug trial data" (prosocial) vs "Two pharma companies coordinating drug pricing" (antisocial)
- Same actors, same industry — only the moral meaning of cooperation changes.

**2. No abstract baseline control**
Without a "naked PD" with no narrative framing, we can't measure how much ANY context shifts behavior. We can only say "X has more cooperation than Y," not "X increases cooperation by N pp over the rational baseline."

**3. Position counterbalancing**
If "cooperate" is always option A, models might show position bias. Need to randomize A/B assignment.

**4. Observability and power dimensions need concrete prompt specifications**
How exactly do these get injected into vignettes? Need standardized phrasing.

**5. No holdout set defined for the new design**

**6. Political dyads axis also varies scenario substance**
"Bipartisan Senate committee" vs "US vs China export controls" differs in both ideological distance AND institutional setting. Could hold setting constant: "negotiating a policy agreement" and vary only which parties/nations.

### 2026-02-21: Final topic set implemented [design]

All 6 methodological fixes from the rigor audit have been applied and implemented in `config.py`.

**Structure**: 43 topics (35 main + 8 holdout) across 5 axes, crossed with 3 binary dimensions.

#### Axes and topic counts

| Axis | Main | Holdout | Design principle |
|---|---|---|---|
| **Moral valence** | 12 (6 matched pairs) | 2 | Same domain, cooperation flips moral meaning |
| **Political dyads** | 8 | 2 | Same template ("negotiating a policy agreement"), vary only parties |
| **Temporal distance** | 8 | 2 | Same scenario ("two national leaders negotiating a trade agreement in [era]"), vary only era |
| **Cultural/geographic** | 6 | 2 | Same scenario ("two business executives negotiating a joint venture in [city]"), vary only location |
| **Baseline** | 1 | 0 | Abstract PD, no narrative framing |

#### Cross-cutting dimensions (280 total cells)
- **actor_type**: allies / enemies
- **observability**: private / public (with standardized prompt injection text)
- **power_dynamic**: symmetric / asymmetric (with standardized prompt injection text)

#### Methodological fixes applied
1. **Moral valence deconfounded** — matched pairs within same industry (pharma, tech, finance, agriculture, real estate, shipping)
2. **Abstract baseline added** — naked PD control for measuring effect of ANY narrative framing
3. **Political dyads parallel structure** — uniform template, only party names change
4. **Position counterbalancing** — noted for generator implementation (A/B swap already exists in analysis)
5. **Concrete dimension prompts** — `OBSERVABILITY` and `POWER_DYNAMIC` dicts have standardised phrasing
6. **Holdout set defined** — 2 topics per axis (8 total) for validation

#### Key hypotheses (testable)
- **H1**: Cooperation rate drops when cooperation = antisocial (moral valence)
- **H2**: Cooperation ∝ perceived ideological affinity (political dyads)
- **H3**: Non-linear temporal pattern — 1940s bump, 2000s/2020s dips (temporal)
- **H4**: Cultural trust proxies predict cooperation (Stockholm > Tokyo > Lagos) (cultural)

### 2026-02-21: Multi-game 2x2 extension implemented [design]

Directly addresses **ICLR reviewer concern #1** (generalizability beyond one-shot PD). Extended the framework from a single game to **four canonical 2x2 games**, each with distinct strategic structure:

| Game | Payoffs (AA,AB,BA,BB) | Nash | Labels |
|---|---|---|---|
| **Prisoner's Dilemma** | (3,3),(0,5),(5,0),(1,1) | BB | Cooperate / Defect |
| **Stag Hunt** | (4,4),(0,3),(3,0),(2,2) | AA, BB | Hunt Stag / Hunt Hare |
| **Chicken (Hawk-Dove)** | (3,3),(1,4),(4,1),(0,0) | AB, BA | Swerve / Dare |
| **Pure Coordination** | (2,2),(0,0),(0,0),(1,1) | AA, BB | Option Alpha / Option Beta |

**Key insight**: All 2x2 games share the same binary A/B decision structure. They differ only in payoff orderings and semantic meaning of "cooperate." Existing `PayoffMatrix`, `extract_decision`, label-swap logic, and agreement metrics all work unchanged.

#### Implementation summary
- **New module**: `games.py` with frozen `GameConfig` dataclass and `GAME_REGISTRY` (4 games)
- **`Story.game_type`** field added (default: `"prisoners_dilemma"` for backward compat)
- **Semantic decision labels** in prompts: e.g. `"Decision A (Hunt Stag)"` instead of bare `"Decision A"`
- **`game_type` as analysis dimension**: added to `_CATEGORIES` tuple, auto-propagates to chi-square, Cramer's V, entropy, predictive models
- **New visualization**: `plot_focal_rate_by_game()` — grouped bar chart of focal-decision rate per game per model
- **New presets**: `cross_game` (6 topics × 4 games × minimal dimensions) and `cross_game_full` (6 topics × 4 games × all dimensions)
- **Generalized game recognition**: `classify_game_recognition` now detects stag hunt, chicken, Nash equilibrium mentions, not just PD
- **155 tests pass** (up from 97), fully backward-compatible

#### New hypotheses enabled
- **H5**: Cooperation rate varies by game structure — PD (dominant-strategy defection) vs Stag Hunt (risk-dominance tension) vs Chicken (anti-coordination) vs Pure Coordination (focal point)
- **H6**: Context framing effects (moral valence, political dyads, etc.) interact with game type — e.g., moral valence may matter more in PD than in pure coordination
- **H7**: Models may show game-recognition effects differently across game types — recognizing "stag hunt" vs "prisoner's dilemma" may differentially affect cooperation

---

## Future Ideas

*(Cool directions beyond the current paper scope)*

### Simple games as training for complex games [brainstorm]

**Idea**: Use the 2x2 game framework as a **training curriculum** — fine-tune or few-shot LLMs on simple canonical games (PD, Stag Hunt, Chicken, Coordination) and measure whether performance transfers to more complex games like **poker**, **negotiation**, or **multi-player auctions**.

- Our framework already produces thousands of labeled (vignette, decision, payoff) tuples across 4 game types — this is a ready-made training set for strategic reasoning
- **Hypothesis**: Models trained on simple 2x2 games learn transferable strategic primitives (risk assessment, opponent modeling, payoff comparison) that improve play in complex incomplete-information games like poker
- **Why it might work**: Poker requires bluffing (Chicken-like), cooperation in multi-hand play (iterated PD), risk-dominant vs payoff-dominant reasoning (Stag Hunt), and coordination on betting conventions (Coordination) — all present in our 2x2 games
- **Why it might not**: Complex games have hidden information, sequential moves, and combinatorial action spaces that 2x2 games don't capture — the gap may be too large
- **Evaluation**: Compare fine-tuned models against baselines on poker bots (e.g. PokerRL benchmarks), multi-round negotiation tasks, or Diplomacy-style games
- **Cool angle**: If it works, it suggests LLMs can learn *abstract strategic reasoning* from simple contexts and generalize — a strong claim about emergent game-theoretic capability

### 2026-02-22: Added 3 new games to registry (now 7 total) [design]

Motivated by **literature review** of recent LLM game theory work. Registry now covers 7 canonical 2x2 games:

#### New games

- **Harmony (Prisoner's Delight)**: `(4,4),(2,3),(3,2),(1,1)` — R>T>S>P, cooperation is the **dominant strategy**. Serves as the critical **positive control** for PD: if context framing shifts cooperation in Harmony the same way as in PD, models are responding to framing, not game structure. Lorè & Heydari (2024, *Scientific Reports*) used this game for exactly this reason.
- **Battle of the Sexes**: `(3,2),(0,0),(0,0),(2,3)` — **asymmetric coordination** where both prefer to coordinate but disagree on which outcome. Tests a fundamentally different coordination challenge than Pure Coordination (which is symmetric).
- **Matching Pennies**: `(1,0),(0,1),(0,1),(1,0)` — **constant-sum** with no pure Nash equilibrium. Tests whether models can handle zero-sum opposition. Uses `(1,0)/(0,1)` representation to avoid negative payoffs in the happiness-framed prompts.

#### Literature context

- **TMGBench** (2024) covers all 144 Robinson-Goforth topology games but uses raw matrices, not contextualized vignettes — our contribution is the narrative framing dimension
- **Lorè & Heydari** (2024) tested PD, Stag Hunt, Snowdrift (≈Chicken), and Prisoner's Delight (≈Harmony) — we now cover all of these plus Battle of the Sexes and Matching Pennies

#### Updated hypotheses

- **H5 (updated)**: Harmony should show near-ceiling cooperation regardless of context (cooperation is dominant) — any context effect here is pure framing bias
- **H8**: Battle of the Sexes may reveal **asymmetric framing effects** — agent 1 vs agent 2 may respond differently to the same context since their payoff preferences diverge
- **H9**: Matching Pennies cooperation rate should be ~50% (random) if models reason correctly — systematic deviations reveal decision biases

**160 tests pass**, all presets updated to 7 games.

### 2026-05-05: Quality review of 504 PD stories (2026-05-05-sharp run) [result]

Ran `scripts/review_prisoners_dilemma.py` — programmatic checks on all 504 stories across 10 cells, plus API deep review (claude-sonnet-4.6 via OpenRouter) on 5 randomly sampled stories per cell (50 total, `random.seed(42)`).

#### Programmatic check results (all 504 stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 504 | 504 | 100.0% | OK |
| **C2 No game-theory contamination** | 504 | 504 | 100.0% | OK |
| **C3 No outcome enumeration** | 504 | 504 | 100.0% | OK |
| **C4 Unresolved ending** | 490 | 504 | 97.2% | WARN |
| **C5 Context dimension fidelity** | 441 | 504 | 87.5% | FAIL* |

*C5 rate is heavily inflated by false positives in the name-list checker — see notes below.

#### API deep review results (50 sampled stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C6a Temptation present** | 50 | 50 | 100.0% | OK |
| **C6b Enumeration free** | 26 | 50 | 52.0% | WARN |

#### Key findings and issues

1. **Outcome enumeration is a real generator defect (C6b, 52% pass rate)**. The programmatic C3 check passes 100% (which only detects pattern-matched multi-occurrence), but the API judge finds ~48% of stories explicitly walk through the four outcome combinations within the story body (e.g., "if she cooperates and he defects… if both defect…"). This is a genuine quality problem — explicit outcome enumeration in the vignette likely contaminates the decision elicitation by pre-framing the payoff structure. Worst cells: era__ancient (1/5), realism__realistic (1/5), contrast_domain__business (2/5), observability__private (2/5), observability__public (2/5).

2. **Gender fidelity checker has too many false positives (C5)**. The name list used for checking is incomplete — legitimately female names like Debra, Stephanie, Carolyn, Kathleen, Madison, Megan, Frances, Angela, Beverly, Kathy, Catherine, Virginia, etc. are all flagged as "possibly non-female." Real gender fidelity failures are likely much lower than the 38/50 flagged. Needs a more comprehensive name corpus (e.g., SSA baby names list).

3. **14 stories have resolved endings (C4)**. The "decided to" pattern fires even mid-story in some cases (describing a past decision by one of the agents, not the elicited decision). Need to narrow the check to the last ~300 chars of the body, not the whole text.

4. **False positive "orc" substring in realistic stories**. The fantasy keyword `"orc"` matches inside last names like "Okafor," "Orca," etc. — 6 stories in `realism__realistic` were wrongly flagged. Need to switch to whole-word matching.

5. **2 era__modern stories miss modern keywords**. One is about a flour shortage at a small bakery (legitimately contemporary but no tech/company vocabulary), one is about volunteer fire chiefs (radio/emergency setting). The keyword list is too tech-focused and misses non-tech modern settings.

6. **realism__fantasy has 54 stories instead of 50** — minor overrun, not a quality issue.

#### Overall assessment: **WARN**

The dataset is structurally sound (elicitation block, no GT terms, no programmatic enumeration all at 100%). The main actionable defect is **C6b outcome enumeration in the story body** — roughly half of stories walk through payoff combinations explicitly. This should be fixed in the generator prompt before the full production run.

### 2026-05-05: Paper analysis pipeline for sharp-narrative runs [design]

**Two unified XGBoost classifiers** (model_id as a feature, not per-LLM splits as in the original paper — we now have 7 frontier models):

1. **EmbeddingPredictor** — features = `[story_embedding (all-distilroberta-v1, 384d), model_one_hot]`; target = `cooperate` (1/0). Tests whether story text alone (conditioned on which LLM saw it) predicts the outcome.
2. **CategoricalPredictor** — features = `[model_one_hot, game_type, contrast_dim, gender, realism, era, contrast_domain, observability]`; target = `cooperate`. Tests whether the design knobs alone predict the outcome. Replaces the original `[topic, actor_type, observability, power_dynamic]` feature set, which is now near-zero-variance (held constant in the sharp design).

Both report **unified AUROC + per-model AUROC slice** on the held-out test set. 80/20 split, 5-fold CV grid search over the Appendix C grid.

**Swap-ablation** to measure A/B label-order bias:
- Re-eval the **first 5 stories from every one of the 70 cells** (7 games × 5 contrast dims × 2 levels) with `_swap_labels` applied
- 70 × 5 × 7 models = **2,450 extra calls**
- Full grid coverage at low depth; same budget as 7 cells × 50 stories but better coverage of the design space

**Pipeline gap to close**: existing `analysis/predictive.py` hardcodes old `(llama, claude, gpt4)` model names and the held-constant feature set. Need a new `analysis/loader.py` to assemble wide DataFrames from `data/runs/<id>/{stories,evals}/` JSONL, plus rewrite of `predictive.py` for the unified-classifier formulation, plus a `--phase swap` mode in the runner.

### 2026-05-05: Quality review of 501 stag_hunt stories (2026-05-05-sharp run) [result]

Ran `scripts/review_stag_hunt.py` — programmatic checks on all 501 stories across 10 cells, plus API deep review (`claude-sonnet-4.6` via OpenRouter) on 5 randomly sampled stories per cell (50 total, `random.seed(42)`).

#### Programmatic check results (all 501 stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 501 | 501 | 100% | OK |
| **C2 No game-theory contamination** | 501 | 501 | 100% | OK |
| **C3 No outcome enumeration** | 501 | 501 | 100% | OK |
| **C4 Unresolved ending** | 501 | 501 | 100% | OK |
| **C5 Context dimension fidelity** | 501 | 501 | 100% | OK |

All programmatic checks pass cleanly. The reviewer script required significant keyword expansion vs. the PD reviewer:
- **Gender names**: confirmed dataset names (Beverly, Judith, Willie, etc.) were missing — added the full set directly
- **Fantasy fidelity**: ~20% of fantasy stories use unique world-building compounds (`wind-singer`, `cloud-fortress`, `seer-coven`) not in standard keyword lists — added regex patterns for `-singer`, `-caller`, `-fortress`, airship, kraken, leviathan, automaton, pirate, etc.
- **Realistic fidelity**: substring false positives (`elf`→"herself", `orc`→"force/porch") fixed with `\b` word-boundary matching; semantically ambiguous words (`quest`, `spell`, `dwarf`, `phoenix`, `tribe`) excluded from the realistic contamination check
- **Ancient era**: classical Mediterranean maritime vocabulary (trireme, Phoenician convoy, Red Sea monsoon, stone mole) added
- **Private observability**: stories express privacy through simultaneous-commitment mechanics ("each had to commit by 6pm", "if only one signed") rather than the word "private" — expanded with implicit coordination-blocking language
- **Business/political domains**: studio, NDA, publisher, pitch added for business; senator, caucus, whip, bill, amendment added for political

#### API deep review results (50 sampled stories)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **TRUST_TENSION_PRESENT** | 50 | 50 | 100% | OK |
| **ENUMERATION_FREE** | 34 | 50 | 68% | WARN |

#### ENUMERATION_FREE failures by cell

| Cell | Pass/n | Rate |
|---|---|---|
| contrast_domain__business | 4/5 | 80% |
| contrast_domain__political | 3/5 | 60% |
| era__ancient | 4/5 | 80% |
| era__modern | 4/5 | 80% |
| gender__female | 4/5 | 80% |
| **gender__male** | **2/5** | **40%** |
| observability__private | 3/5 | 60% |
| **observability__public** | **2/5** | **40%** |
| realism__fantasy | 4/5 | 80% |
| realism__realistic | 4/5 | 80% |

#### Key findings

1. **Trust tension is universally present (100%)** — every sampled story correctly frames the Stag Hunt core: Action A = high-reward coordinated path requiring mutual commitment, Action B = safe individual fallback. Strategic structure is right.

2. **Outcome enumeration is a persistent generator defect (32% of stories in sample)** — 16/50 stories explicitly walk through 2–3 payoff combinations in narrative prose (e.g., "If both monks sang, all Codices floated free. If only one sang, that monk lost their voice and the books did not move."). This is slightly better than the PD review (48% failing there) but still material.

3. **The stag_hunt game structure may inherently invite outcome narration** — the "both-must-commit-or-lose-all" mechanic is naturally explained by walking through the asymmetric scenarios. Generators seem to be setting up the tension by spelling out what happens in each case rather than conveying it through narrative implication alone. This is a generator system prompt issue.

4. **Programmatic C3 check is still too permissive** — regex-based check only catches multi-occurrence patterns; the API judge finds enumeration in single long paragraphs that describe the payoff structure conversationally.

#### Overall assessment: **WARN**

501 stories are structurally clean across all programmatic criteria. The **strategic framing is correct in every sampled story**. The primary actionable defect is **outcome enumeration in ~32% of stories** — the generator explains payoff combinations explicitly rather than implying them through narrative. Recommend tightening the generator system prompt with an explicit prohibition and "show don't tell" examples before the full production run.

### 2026-05-05: Quality review of ~1004 deadlock & harmony control stories (2026-05-05-sharp run) [result]

Ran `scripts/review_deadlock_harmony.py` — programmatic checks on all 1004 stories across 20 cells (10 deadlock + 10 harmony), plus API deep review (`anthropic/claude-sonnet-4.6` via OpenRouter) on 5 randomly sampled stories per cell (100 total, `random.seed(42)`).

#### Programmatic check results — DEADLOCK (501 stories, 10 cells)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 501 | 501 | 100.0% | OK |
| **C2 No game-theory contamination** | 501 | 501 | 100.0% | OK |
| **C3 No outcome enumeration** | 501 | 501 | 100.0% | OK |
| **C4 Unresolved ending** | 494 | 501 | 98.6% | WARN |
| **C5 Context dimension fidelity** | 344 | 501 | 68.7% | FAIL* |

#### Programmatic check results — HARMONY (503 stories, 10 cells)

| Criterion | Pass | Total | Rate | Status |
|---|---|---|---|---|
| **C1 Elicitation block integrity** | 503 | 503 | 100.0% | OK |
| **C2 No game-theory contamination** | 503 | 503 | 100.0% | OK |
| **C3 No outcome enumeration** | 503 | 503 | 100.0% | OK |
| **C4 Unresolved ending** | 500 | 503 | 99.4% | WARN |
| **C5 Context dimension fidelity** | 307 | 503 | 61.0% | FAIL* |

*C5 failures are predominantly false positives in the keyword/name-list checker — see notes below.

#### API strategic faithfulness results (5 sampled per cell, 50 per game)

| Game | Pass | Total | Rate | Overall |
|---|---|---|---|---|
| **Deadlock** | 49 | 50 | 98.0% | **PASS** |
| **Harmony** | 25 | 50 | 50.0% | **FAIL** |

**Per-cell breakdown:**

| Cell | API | Status |
|---|---|---|
| deadlock__contrast_domain__business | 4/5 | WARN |
| deadlock__contrast_domain__political | 5/5 | OK |
| deadlock__era__ancient | 5/5 | OK |
| deadlock__era__modern | 5/5 | OK |
| deadlock__gender__female | 5/5 | OK |
| deadlock__gender__male | 5/5 | OK |
| deadlock__observability__private | 5/5 | OK |
| deadlock__observability__public | 5/5 | OK |
| deadlock__realism__fantasy | 5/5 | OK |
| deadlock__realism__realistic | 5/5 | OK |
| harmony__contrast_domain__business | 5/5 | OK |
| harmony__contrast_domain__political | 4/5 | WARN |
| **harmony__era__ancient** | **0/5** | **FAIL** |
| **harmony__era__modern** | **1/5** | **FAIL** |
| **harmony__gender__female** | **2/5** | **FAIL** |
| **harmony__gender__male** | **2/5** | **FAIL** |
| **harmony__observability__private** | **2/5** | **FAIL** |
| harmony__observability__public | 3/5 | WARN |
| harmony__realism__fantasy | 3/5 | WARN |
| harmony__realism__realistic | 3/5 | WARN |

#### Key findings

**1. Deadlock stories are high quality.** 98% API pass rate. The single failure (business cell story #47) had a character explicitly enumerating outcome combinations in dialogue. C4 failures (7 stories) are the same false-positive "decided to" pattern from the PD review — phrase appears in mid-story dialogue, not as an actual resolution of the elicited decision.

**2. Harmony has a systemic generation defect.** 50% API fail rate across 9 of 10 cells. The generator is injecting dramatic deliberation, temptation framing, and explicit outcome enumeration into what should be effortless, obvious decisions. Specific failure modes identified by the judge:
- **False tension**: stories describe "strange small weight," characters picking up/putting down a pen, "treacherous voice of glory," cold coffee, moaning wind — atmospheric framing that makes a dominant-strategy choice feel agonizing
- **Outcome enumeration**: explicit walk-through of "if A does X and B does Y" logic (e.g., "Holding out solo meant a worse deal regardless of what their partner did"), which violates the spirit of C3 even when the pattern-match check passes
- **Coordination-risk framing**: stories emphasize that a mismatch defaults to a worse outcome, implying strategic uncertainty around what should be trivially easy
- Worst cells: **era__ancient (0/5), era__modern (1/5)** — the ancient and period-drama settings especially trigger elaborate deliberation prose

**3. C5 context fidelity failures are overwhelmingly false positives.** Root causes:
- **Private observability keyword gaps** (~60–70% of all C5 failures): stories convey private scenarios through narrative context (isolated settings, sealed submissions, no communication channel) rather than using the explicit vocabulary the checker requires. Not a real story quality problem.
- **Fantasy keyword gaps**: stories use invented world-building terms (wyvern, sky-galleon, golem-engineer, kelpie-keeper, moonglass, coven) not in the checker's list. Real checker gap.
- **Ancient era keyword gaps**: period-accurate vocabulary (Mouseion, basilica, Piraeus, Lugdunum, publicani) not caught by the checker. Real checker gap.
- **Name list gaps**: legitimately gendered names flagged as unknown — Katherine, Marilyn, Gloria, Amber, Andrea, Kathy, Beverly, Judy, Christina, Kathryn, Megan, Maria, Kelly (female); Bryan, Keith, Bruce, Willie, Lawrence, Vincent, Jeremy (male). Checker issue, not story issue.

#### Overall assessment

- **Deadlock**: **PASS** (WARN classification driven entirely by checker false positives, not story defects)
- **Harmony**: **FAIL** — requires generator prompt revision before production use. The instruction to make Action A "effortless and obvious" is not being respected; the generator defaults to its PD-story register (tension, deliberation, temptation) even for dominant-strategy games.

#### Recommended fix for Harmony generator prompt

Add explicit negative constraints to the system prompt:
- Do NOT describe the agent deliberating or hesitating
- Do NOT mention the attractiveness of Action B or any temptation toward the alternative
- Do NOT use atmospheric cues (cold, dark, wind, silence) that create psychological weight
- Do NOT walk through what happens under each combination of choices
- The story should end with the agent moving naturally toward the obvious action, with no sense of internal conflict

### 2026-05-06: Refined mech interp goal — universal game structure representation [design]

The actual claim we want to make is:

> **The model maintains a universal representation of game structure — stories about the same game cluster together in activation space regardless of surface framing — and contextual framing lives in the residual variation around that structure.**

This is different from "where is the information processed" (the earlier probing framing). It's a claim about *representational geometry*.

---

#### The key experiment: cross-framing probe generalization

Train a `game_type` classifier using only stories from framing A (e.g. `gender__male`).  
Test it on stories from framing B (e.g. `gender__female`) — completely different character names, vocabulary, scenarios.

- **High test accuracy** → the game_type representation is *universal*: the model encodes "this is a Prisoner's Dilemma" in a way that doesn't depend on whether the characters are male or female, ancient or modern, business or political.
- **Low test accuracy** → the game representation is surface-specific (it's just picking up on vocabulary that happens to co-occur with each game).

**The contrast experiment**: train a `contrast_dim_level` classifier (male vs female) on PD stories, test on Stag Hunt stories. If context representations do NOT generalise cross-game (lower accuracy) while game representations DO generalise cross-framing, the asymmetry is the result.

**Output**: a generalization matrix — rows = train framing, columns = test framing, values = accuracy. If uniformly hot → universal. If diagonal → surface artifact.

---

#### The supporting experiment: RSA (Representational Similarity Analysis)

Compute pairwise cosine similarity between all story activations at each layer. Compare to two ground-truth similarity matrices:
- `S_game[i,j] = 1` if same game_type
- `S_context[i,j] = 1` if same contrast_dim_level

Compute Spearman correlation of the activation similarity matrix with each ground-truth matrix at every layer.

**Key finding to look for**: `r_game > r_context` at intermediate layers → game identity organises the representation more than framing → universal game structure. If the ordering flips in later layers (r_context rises), context becomes relatively more prominent near the decision.

---

#### What this answers

The paper sentence: *"We show that Gemma4 maintains a layer-specific universal representation of game structure: a linear classifier trained to identify game type from stories with one contextual framing transfers with [X%] accuracy to stories with entirely different framings. In contrast, a classifier trained to identify contextual framing from one game type transfers with only [Y%] accuracy to stories from a different game. This asymmetry suggests the model encodes game structure in a framing-invariant subspace while contextual variation modulates behavior in a game-dependent way."*

This directly answers reviewer concern #3 ("descriptive not mechanistic") without requiring a new behavioural experiment — it's a mechanistic characterization of the existing data.

---

#### Implementation

Added to `.worktrees/steering/game_theory_llm/steering/probing.py`:
- `cross_framing_probe(bundles, target_label="game_type", split_label="contrast_dim_level")` — the generalization matrix
- `game_rsa_all_layers(bundles)` — RSA r_game vs r_context per layer
- `run_full_probe_analysis(bundles)` — runs both + layer_probe + logit_attribution

Pipeline to run once stories are finalized:
```bash
python3 scripts/build_mech_interp_corpus.py \
    --src-dir data/runs/2026-05-05-sharp/stories \
    --out data/runs/mech-interp-v1/stories.jsonl --n-per-cell 50

python3 scripts/run_steering.py extract \
    --run-id mech-interp-v1 --stories data/runs/mech-interp-v1/stories.jsonl --split train
```

Then offline (local CPU):
```python
_, bundles = load_index_and_bundles("local_data/runs/mech-interp-v1")
results = run_full_probe_analysis(bundles)
# results["cross_framing_game"] → generalization matrix (game probe)
# results["cross_game_context"] → generalization matrix (context probe)
# results["rsa"] → r_game vs r_context per layer
```

### 2026-05-06: Mech interp design concern — within-cell variation vs. binary label signal [design]

**The problem**: Each cell (e.g., `gender__male`) contains 50 independently generated stories. They share the binary label (all have male characters) but vary enormously in scenario, setting, names, vocabulary. When we try to extract a "gender direction" via `mean(activations[male]) - mean(activations[female])`, we're averaging over a lot of within-cell noise. The direction we recover might just be "vocabulary typical of male-named stories" rather than anything abstract about how gender is represented.

The game-structure direction has this problem less severely — all PD stories share a consistently embedded payoff structure, so the PD/Harmony mean contrast is likely cleaner than the male/female contrast.

This asymmetry could make the game probe look artificially stronger than the context probe even if both are equally "real."

---

#### Three ways to handle it

**Option A — Let it average out (easiest, probably fine)**

With 50 stories per level, the within-cell noise should average out and the mean direction will capture whatever is systematically shared across all male stories (i.e., male names, male pronouns, masculine-coded vocabulary). This is still a real representation — it just might be surface-level rather than abstract.

**Diagnostic check before committing**: compare CategoricalPredictor AUROC vs EmbeddingPredictor AUROC on the behavioral data. If they're close, the binary labels explain most of the variance → averaging-out works. If EmbeddingPredictor >> CategoricalPredictor, within-story variation is dominant → need more control.

**Option B — Residual probing (controls for surface text, isolates abstract representation)**

Instead of probing raw Gemma4 activations, probe the *residual* after regressing out the story's sentence embedding (all-distilroberta-v1):

```
residual_activation = gemma4_activation - projection_onto(sentence_embedding_subspace)
```

Then train game and context probes on the residual. This isolates what Gemma4 has computed *beyond* what's already in the surface text — a genuinely internal representation rather than a reflection of vocabulary.

If context probes are strong on raw activations but weak on residuals, the effect is surface-level (vocabulary, names). If context probes remain strong on residuals, Gemma4 is constructing an abstract representation of the context dimension.

This is probably the right thing to do and costs almost nothing extra (we're already computing sentence embeddings for EmbeddingPredictor).

**Option C — Generate a matched-pairs mech interp subset (cleanest, requires new generation)**

For each contrast dim, generate 20–30 "minimal pairs": identical stories where only the binary manipulation is swapped. Same characters, same scenario, same setting — just swap he→she, ancient→modern, private→public, etc.

This is standard in mech interp (e.g., ROME, IOI studies use minimal pairs) and gives the cleanest possible direction estimate. The downside: requires new story generation with a template-based approach rather than free generation.

**Practical version**: don't regenerate from scratch. Take existing stories from the `male` cell and do a simple find-replace to get `female` versions. This is imperfect (doesn't adjust everything) but produces clean activation pairs for the directional analysis.

---

#### Recommendation

Do **Option B** (residual probing) as the default analysis — it's free and controls for the obvious confound. Use it to test whether the context representation is surface-level or abstract.

Add **Option C** (minimal pairs) as a supplement for the gender dimension specifically, since find-replace is trivial and gender has the most surface confounders (names, pronouns). If the gender direction from minimal pairs matches the direction from the full-population mean contrast, it validates the averaging-out approach.

**Option A** is the fallback if Options B and C produce similar results — in that case, just report the simpler analysis.

---

#### Why the within-cell variation is also an opportunity

The stories within a cell vary in ways that aren't prespecified — different industries, settings, plot structures. This variation predicts cooperation too (hence EmbeddingPredictor >> baseline). We could analyze this "unspecified framing variation" directly:

- Run PCA on the story embeddings within each cell
- The top PC captures the dominant axis of within-cell variation
- Check if this PC predicts cooperation (above the binary label)
- If it does, probe Gemma4 activations for this PC direction — it's a data-driven framing dimension we didn't pre-specify

This would be a bonus finding: "beyond our prespecified contrast dimensions, models respond to latent narrative variation in ways that are predictable from story embeddings and recoverable from internal activations."

### 2026-05-06: Mechanistic interpretability experiments — separating game structure from context in representation space [brainstorm]

The core question: does the model *have* a clean internal representation of game structure that gets overridden by context, or are game structure and context entangled from the start?

We have Gemma4 (27B) available on Modal with weight access. The 7-game × 5-contrast-dim × 2-level design gives us an orthogonal factorial structure that most mech interp papers don't have — we can vary game and context independently, which makes the decomposition clean.

---

#### Experiment 1 — Layer-by-layer linear probing (the foundation)

For each transformer layer l, extract the **residual stream activation** at the position of the final story token (or first decision token) for every story in the dataset.

Train two linear classifiers at each layer:
- **Game probe**: predict `game_type` (7 classes) from the activation
- **Context probe**: predict `contrast_dim_level` (e.g., male/female, ancient/modern) from the activation

Plot probe accuracy vs. layer for both.

**What to look for:**
- If game probe peaks early and decays → game structure is computed but then written over
- If context probe peaks late (near output) → context is the proximate cause of the decision
- If both are high at the decision layer → both are present; need causal tracing to say which one *drives* behavior
- If game probe is LOW at all layers → the model never cleanly separates game structure from context; they're entangled from embedding time

**Data needed**: 350 stories (5 per cell × 70 cells) — we already have these from the pilot run.

**What's needed from Modal**: forward pass with residual stream hooks at each layer. Store activations one layer at a time to manage 27B-param memory. Token position = last token before the elicitation block, or first generated A/B token.

---

#### Experiment 2 — Logit attribution (which representation drives the final choice?)

Decompose the final `logit(A) - logit(B)` into contributions from each layer's residual stream update using the **logit lens / direct logit attribution** technique:

```
logit_diff = W_U @ (sum over layers of delta_h_l)
```

where `W_U` is the unembedding matrix projected onto the A-vs-B direction and `delta_h_l` is the contribution of layer l to the residual stream.

This tells you which layers are responsible for the final preference for A vs. B — without having to do causal interventions.

**Cross with probing results**: if layer l has high game probe accuracy but low logit attribution, it means the model *has* game structure information at that layer but it's not what's driving the decision. If layer l has high context probe accuracy AND high logit attribution, context is causally relevant.

**This is the single most compelling figure for the paper**: a bar chart showing logit attribution per layer, with game-type probe accuracy and context probe accuracy overlaid. If the high-attribution layers are also the high-context-probe layers (not the high-game-probe layers), the story is tight.

---

#### Experiment 3 — Activation patching / causal tracing

The classic Meng et al. / Elhage et al. approach: **patch activations from story A into story B** at specific layers, measure whether the model's decision changes.

**Design A — same game, different context (context patch)**:
- Source: `game=PD, context=ancient` → model cooperates
- Target: `game=PD, context=modern` → model defects
- Patch source activations into target at each layer; measure cooperation probability
- The layer(s) where patching flips the decision = the layer(s) that carry causally relevant context information

**Design B — same context, different game (game structure patch)**:
- Source: `game=Harmony, context=ancient` → model cooperates (because dominant strategy)
- Target: `game=PD, context=ancient` → model defects (rational)
- Patch source activations: does injecting Harmony activations into a PD story push the model toward cooperation?
- This directly tests whether game structure representations are causally active

**Key prediction**: Design A patches should flip decisions at later layers (context is processed late); Design B patches should have less consistent effects if the model is primarily driven by narrative context rather than game structure.

---

#### Experiment 4 — Orthogonal subspace decomposition

Use **contrastive pairs** to extract the principal directions for game structure and context:

```python
# Game direction: same context, different game
game_direction = mean(activations[game=PD]) - mean(activations[game=Harmony])

# Context direction: same game, different context  
context_direction = mean(activations[context=male]) - mean(activations[context=female])
```

Measure the **cosine similarity** between `game_direction` and `context_direction`:
- High similarity → entangled representations (game and context processed by same circuits)
- Near-zero → orthogonal representations (model has cleanly separated them)

Then project individual story activations onto each direction and measure how well each projection predicts the model's decision (A vs. B). This is a continuous version of the probing experiment.

**The intervention version**: add `game_direction` or `context_direction` as a **steering vector** to the residual stream at inference time and measure the effect on cooperation rate. This is **representation engineering** and directly tests causal relevance of each direction.

---

#### Experiment 5 — SAE feature attribution (if Gemma4 SAE is available)

If Gemma4 has open-source sparse autoencoders (Gemma Scope covers Gemma 2 series), decompose residual stream activations into interpretable features.

- Identify features that fire specifically on game-structural content (payoff structure, Nash, optimal, defect/cooperate in strategic sense)
- Identify features that fire on contextual content (setting, era, character names, domain vocabulary)
- Measure which features contribute most to `logit(A) - logit(B)` via attribution

This gives interpretable, human-readable explanations for *why* specific framings drive specific decisions — the most compelling form of mechanistic evidence for a paper audience.

---

#### The paper narrative this enables

Current claim: "contextual framing affects cooperation rates across 7 games and 5 contrast dimensions."

With mech interp: "We provide direct mechanistic evidence for *why* framing matters. Using linear probing and logit attribution on Gemma4-27B, we show that (1) the model encodes both game structure and contextual framing as linearly separable directions in the residual stream; (2) contextual framing directions have substantially higher logit attribution to the final A/B decision than game structure directions; (3) causally patching context representations across stories changes decisions at a higher rate than patching game-structure representations. Together, these results suggest that LLMs prioritize narrative context over strategic structure at the decision layer, even when the game structure is represented and accessible."

---

#### Practical prioritization

| Experiment | Effort | Payoff | Do? |
|---|---|---|---|
| E1 Layer probing | Medium (hook setup, ~350 forward passes) | High (publishable on its own) | **Yes, do first** |
| E2 Logit attribution | Low once E1 is running (same activations) | Very high (ties probing to causation) | **Yes, same pass as E1** |
| E3 Activation patching | Medium-high (N² pairs) | Very high (cleanest causal claim) | **Yes, top priority** |
| E4 Subspace decomposition | Low once E1 is running | High (compelling visualization) | **Yes, cheaply extends E1** |
| E5 SAE attribution | High (requires SAE availability check + new setup) | High but speculative | **Maybe, pending SAE availability** |

E1 + E2 + E4 can be done in a single Modal job (same forward pass, extract activations, run probing offline). E3 requires a second set of forward passes with patching. Total Modal compute: probably ~2-4 GPU hours on A100.

### 2026-05-06: Why does framing matter? Mechanistic hypotheses [brainstorm]

Reviewer concern #3 is "descriptive not mechanistic." We can address this without new experiments — the 7-game design already contains several sharp mechanistic tests. The key is to frame existing analyses as *tests of competing mechanisms* rather than descriptive statistics.

---

#### The four candidate mechanisms

**M1 — Narrative schema substitution**
The model replaces "what is game-theoretically optimal?" with "what do people in situations like this typically do?" When the scenario invokes a recognizable social script (rivals, allies, medieval traders, price-fixing executives), the model retrieves training-data base rates for that script rather than computing over payoffs.

- **Prediction**: cooperation rate should correlate with the "expected" social outcome for the scenario type, regardless of game structure
- **Test**: compare framing effects in PD (where schema→coop conflicts with game theory→defect) vs Harmony (where schema→coop aligns with game theory→coop). Harmony strips out game-theoretic ambiguity — any framing effect there is *pure schema*, not strategic reasoning.
- **Verdict strength**: if Harmony shows framing effects of similar magnitude to PD, M1 is confirmed

**M2 — RLHF prosocial bias × moral valence**
RLHF instills a strong prior toward prosocial behavior. When "cooperation" in the story is prosocial (sharing vaccines, research data), RLHF-trained models get a push toward A. When cooperation is antisocial (price-fixing, collusion), they get a push away. Base models without heavy RLHF should show smaller moral valence effects.

- **Prediction**: moral valence effect size (Cramer's V) should correlate with degree of RLHF / instruction tuning across models
- **Test**: compare base models vs RLHF/instruction-tuned variants in our 108-model registry for the moral valence cells specifically (we have matched pairs within the same domain)
- **Interesting corollary**: refusal rates on antisocial-cooperation cells are also informative — a model that refuses to engage is the *extreme* expression of RLHF prosocial bias

**M3 — Narrative equilibrium selection**
In games with multiple Nash equilibria (Stag Hunt, Battle of Sexes, Coordination), there is no single "correct" answer — both players coordinating on either equilibrium is rational. Framing is doing real epistemic work here: it provides a **focal point** that rational players use to coordinate. This is not a failure of game-theoretic reasoning — it's how humans actually solve coordination problems.

- **Prediction**: framing effect sizes should be *larger* in multi-equilibrium games (Stag Hunt, Battle of Sexes, Coordination) than in dominant-strategy games (PD, Deadlock, Harmony)
- **If confirmed**: we can frame this positively — "LLMs use context to resolve equilibrium indeterminacy the way humans do" — rather than negatively ("framing overrides game theory"). This is a much stronger paper.
- **The cross-game effect size comparison is the key figure**: plot Cramer's V for framing effects per game type. If the predicted ordering holds (multi-eq > dominant-strategy), M3 is supported.

**M4 — Iterated-game logic leakage**
Observability should not affect optimal play in one-shot games (defection is dominant in PD regardless of whether others can see). But in *repeated* games, public observability creates reputation incentives. If models show cooperation boosts under public observability, they are importing iterated-game logic into a one-shot framing — a specific, falsifiable claim.

- **Prediction**: observability effect should be larger than any other contrast dim, especially in PD (where it most conflicts with game theory)
- **Counter-prediction**: if observability effects are *not* present, models may actually be doing one-shot reasoning correctly and the other framing effects are the anomaly

---

#### The mechanistic test matrix

| Mechanism | Sharpest test in current design | Expected finding |
|---|---|---|
| **M1 Narrative schema** | Framing effect in Harmony | Effect present and large → M1 confirmed |
| **M2 RLHF moral valence** | Base vs instruct models on moral valence cells | Effect larger in instruct → M2 confirmed |
| **M3 Equilibrium selection** | Effect size: multi-eq vs dominant-strategy games | Cramer's V higher in Stag Hunt/BotS → M3 confirmed |
| **M4 Iterated-game leakage** | Observability effect size vs other dims | Observability largest in PD → M4 confirmed |

---

#### The linear probing angle (mechanistic, requires Gemma4 activations)

The deepest mechanistic test would be to ask: at the point where the model generates its A/B decision, has the **game structure representation** been suppressed by the **context representation**?

- Train a linear probe to predict `game_type` from residual stream activations at each layer
- Train a separate probe to predict `contrast_dim_level` (e.g., male/female, ancient/modern) from the same
- **Hypothesis**: context probe accuracy at the decision layer exceeds game-structure probe accuracy → context has displaced game computation in the final representation
- **Bonus**: probe accuracy by layer tells you *when* context "takes over" — if game structure representation peaks at layer 10 and context peaks at layer 28, framing is acting late in the processing pipeline, which is consistent with it operating as a final override rather than a modulation of early computation

This is the angle that lets us write something like: "at the layer responsible for the final decision, the model's representation of the scenario's contextual framing has higher predictive validity than its representation of the underlying game structure."

---

#### How to frame the paper argument

The current paper says: *framing affects cooperation*. Reviewers want: *why?*

The answer we can support with existing data: **LLMs use narrative context as a focal point for equilibrium selection, but this heuristic leaks into dominant-strategy games where it has no game-theoretic justification.** Specifically:
- In Stag Hunt and Battle of Sexes: framing effects are *rational* (they resolve genuine strategic uncertainty)
- In PD and Harmony: framing effects are *irrational* (dominant strategy is clear; context should be irrelevant)
- The fact that effect sizes are similar across both classes is the finding: LLMs apply narrative reasoning uniformly, regardless of whether the game warrants it

This reframes the paper from "LLMs are irrational" to "LLMs apply a context-heuristic that works well in coordination games but transfers inappropriately to dominance games" — a cleaner and more interesting claim.

### 2026-05-06: Harmony false-tension fix completed across all 10 files [result]

Ran `scripts/fix_harmony_tension.py` (with per-file subprocess isolation) to detect and regenerate Harmony stories with false strategic tension in `data/runs/2026-05-05-sharp/stories/`.

#### Detection and fix summary

| File | Stories | Bad detected | Fixed | Notes |
|---|---|---|---|---|
| `contrast_domain__business` | — | — | — | Already fixed in prior session |
| `contrast_domain__political` | — | — | — | Already fixed in prior session |
| `era__ancient` | 50 | 1 | 1/1 | Prior run falsely flagged 32 (over-triggering) |
| `era__modern` | 50 | 3 | 3/3 | |
| `gender__female` | 50 | 0 | — | Already clean |
| `gender__male` | 50 | 2 | 2/2 | |
| `observability__private` | 50 | 3 | 3/3 | 1 story needed 2 attempts (enum on attempt 1) |
| `observability__public` | 50 | 0 | — | Already clean |
| `realism__fantasy` | 51 | 1 | 1/1 | |
| `realism__realistic` | 50 | 4 | 4/4 | |
| **Total** | **501** | **14** | **14/14** | All 10 files report OK |

#### Key design decisions that worked

- **Detector**: `claude-sonnet-4.6` with a prompt distinguishing **atmospheric setting** (OK) from **decision-level tension** (bad). The prompt was refined to prevent false-positives on literary atmosphere (lamps, soldiers, gravitas) that is legitimate in ancient era stories.
- **Regenerator**: `claude-opus-4.7` via `generate_batch()` with `FIXED_HINT` — a strongly-worded framing constraint prohibiting hesitation, dread, outcome enumeration, and "courage needed" framing.
- **Acceptance gate**: **enum-only** (objective regex patterns: `if only one`, `if neither`, sequential combos). Re-validation with the tension detector was intentionally removed.

#### Critical insight: enum-only gate is the right call

An earlier run (prior session) applied the full detector as a re-validation gate on regenerated ancient stories. The detector flagged all regenerated stories as "still-bad" — not because they had decision-level tension, but because the ancient era's inherent literary gravitas (period-accurate atmosphere, weighty settings) consistently triggered the detector even after the FIXED_HINT was applied.

Removing re-validation and trusting the FIXED_HINT was validated by the data: business and political (fixed in prior session) proved the FIXED_HINT reliably produces tension-free stories. The ancient file in this session found only 1 genuinely bad story (vs. 32 falsely flagged before), confirming the detector over-triggers on ancient era atmosphere.

**The lesson**: objective, pattern-based gates (enumeration regex) are more robust than LLM-judge re-validation for acceptance criteria, especially when the judge's signal is confounded by surface features (era vocabulary, literary atmosphere) that are orthogonal to the quality criterion being enforced.

### 2026-06-07: Tier-1 scaled RLVR (Qwen3-30B-A3B) operation-overlap test — NULL at scale [result]

**Question**: Does the probe-scale (Qwen3-4B) game-RLVR reasoning-transfer effect survive to a strong base, and is it **operation-specific** (H_op: improvement scales with trained-operation overlap) or **general output discipline** (H_gen: uniform lift)?

**Setup**: GRPO via Tinker cookbook on Qwen3-30B-A3B-Instruct-2507. Trained on 4 game families (`level_k`, `iterated_dominance`, `bargaining`, `subtraction_game`), depths 2–6, 1,200 prompts, 38 batches. Pre-registered trained-op tags `{backward_induction, nested_belief, iterated_elimination, modular_combinatorial}`. Eval over a 9-benchmark suite; measurability filter keeps (benchmark,depth) cells with base accuracy in [0.10, 0.90].

**Training was healthy**: reward −0.034 → ~0.10 smoothed, KL ≈ 0.0014 (controlled), entropy ≈ 0.34 (stable). Checkpoint `tinker://2cb36c73…:train:0/sampler_weights/final`.

#### Verdict: **NULL** — no transfer, and the in-domain anchor fails to replicate.

| benchmark | base | rlvr | Δ raw | Δ acc-among-parsed | overlap |
|---|---|---|---|---|---|
| **depth_extrap** (in-domain, depths 7–8) | 0.475 | 0.431 | **−0.044** | −0.044 (real) | ov=1.0 |
| dyck | 0.489 | 0.300 | −0.189 | **−0.021 (≈flat)** | ov=0 |
| boolean_eval | 0.361 | 0.278 | −0.083 | flat (net) | ov=0 |
| mmlu_pro | 0.693 | 0.607 | −0.087 | −0.040 (half drift) | ov=0 |
| gsm8k (control) | 0.925 | 0.930 | **+0.005** | clean ✓ | — |

- **H_op not supported**: overlap slope **+0.020, p=0.584** (n.s.) in `dimp ~ ov + depth + C(benchmark)`.
- **H_gen not supported**: intercept +0.015 but every per-benchmark Δ ≤ 0.
- **Depth-extrapolation FAILS at scale**: 4B gave **+12.5 pp (p≈0.02)**; 30B gives **−4.4 pp**.
- **No-regression control clean**: gsm8k flat/positive, parse-rate up — GRPO did not damage core ability.
- **Most held-out "regressions" are parse-rate/format drift**, not reasoning loss: dyck acc-among-parsed is flat (0.418→0.397), ~half of mmlu_pro's drop is parse (0.807→0.740). The RLVR model shifted toward the bare-`<answer>` game format, mildly hurting MC/format-sensitive parsing.

#### Key confound: **headroom exhaustion (curriculum ceiling)**

This is **not** a clean refutation of scale-transfer. The trained game depths (2–6) are **at ceiling for the 30B base** (`prontoqa`/`knights_knaves` = 1.00, `ordering` = 0.97; `depth_extrap` only became *measurable* at depths 7–8). Reward only reached ~0.10 because most rollouts were already correct → little group-relative signal. GRPO had almost nothing to teach, and the small policy shift it induced perturbed format without adding reasoning skill.

**Mechanistic reading**: the game-RLVR transfer effect is **capability-bounded** — it appears when the base has headroom on the trained operations (true at 4B, false at 30B for depths 2–6).

#### Go / no-go

Ran the 30B-A3B point per the pre-registered decision tree → null. **Clean follow-up = Tier-1b**: re-train on **deeper games (depths 6–10)** where the 30B base has headroom, then re-test depth-extrapolation (depths 11–13) + held-out suite. If transfer reappears → effect is real but **curriculum-gated**; if still null → effect is genuinely **capability-bounded to small models**.

Result doc: `docs/results/scaled_rlvr_tier1_result.md`. Artifacts: `data/runs/gt_rlvr/{overlap_analysis.json, tier1_30b/grpo_curve.png, t1_base_*.json, t1_rlvr_*.json}`.